# Day 58 Practice: Put the AI Behind One Function

Small example: yesterday's **course-feedback app**. Today an LLM labels each comment as `praise`,
`complaint` or `suggestion`, behind one function, with fallbacks and tests that never need the network.

✅ Exercises 1–4: everyone (about 20 min) · Exercise 5: most of you · Exercise 6: challenge (option A students:
do it, it is your pattern)

Then spend the build time doing the same for **your own** capstone.

**Run the setup cell first.** It gives you `ask_llm`, a `mock_llm` written for this app, `build_prompt` and
yesterday's keyword stub (renamed `keyword_label`, now the fallback).

In [ ]:
# ---------- setup: run this cell, do not change it ----------
import json
import os

import anthropic
from dotenv import load_dotenv

load_dotenv()
MODEL = os.environ.get("LLM_MODEL", "claude-opus-5-5")
REFUSAL_MESSAGE = "[The model declined this request.]"
LABELS = ["praise", "complaint", "suggestion"]


def keyword_label(comment):
    """Day 57 stub, now the fallback: simple keyword rules."""
    text = comment.lower()
    for word in ["please", "could", "should", "would be"]:
        if word in text:
            return {"label": "suggestion", "reason": f"contains '{word}'"}
    for word in ["too ", "lost", "boring", "confusing"]:
        if word in text:
            return {"label": "complaint", "reason": f"contains '{word}'"}
    return {"label": "praise", "reason": "no complaint or suggestion words"}


def build_prompt(comment):
    return ("Label this course-feedback comment as exactly one of: praise, complaint, suggestion.\n"
            'Reply with JSON only: {"label": "...", "reason": "a few words"}\n'
            f"Comment: {comment}")


def mock_llm(prompt):
    """Offline stand-in: apply the keyword rules to the 'Comment:' line and answer in JSON."""
    for line in prompt.splitlines():
        if line.startswith("Comment: "):
            return json.dumps(keyword_label(line[len("Comment: "):]))
    return json.dumps({"label": "praise", "reason": "no comment found"})


def ask_llm(prompt, system="You are a helpful assistant."):
    """Send one prompt to Claude and return its text reply (mock reply when offline)."""
    load_dotenv()
    if os.environ.get("COURSE_OFFLINE") == "1" or not os.environ.get("ANTHROPIC_API_KEY"):
        return mock_llm(prompt)
    client = anthropic.Anthropic()
    response = client.beta.messages.create(
        model=MODEL,
        max_tokens=16000,
        system=system,
        messages=[{"role": "user", "content": prompt}],
        betas=["server-side-fallback-2026-07-01"],
        fallbacks="default",
    )
    if response.stop_reason == "refusal":
        return REFUSAL_MESSAGE
    return "".join(block.text for block in response.content if block.type == "text")


print(ask_llm(build_prompt("Please record the lectures.")))

## Exercise 1: Which replies are usable?

Here are six **hand-written** replies of the kind models sometimes send. For each, predict:
(a) does `json.loads` succeed? (b) can the app *use* it (a dict with a label from `LABELS` and a reason)?
Write your predictions in the table, then run the cell, which only checks (a).

| | Reply | (a) JSON? | (b) usable? |
|-|-------|-----------|-------------|
| 1 | `{"label": "praise", "reason": "positive about labs"}` | | |
| 2 | `The label is praise.` | | |
| 3 | `{"label": "angry", "reason": "upset student"}` | | |
| 4 | `{"label": "complaint"}` | | |
| 5 | the JSON of a suggestion inside a ```` ```json ```` code fence | | |
| 6 | `{"label": "Complaint", "reason": "too fast"}` | | |

In [ ]:
replies = [
    '{"label": "praise", "reason": "positive about labs"}',
    "The label is praise.",
    '{"label": "angry", "reason": "upset student"}',
    '{"label": "complaint"}',
    '```json\n{"label": "suggestion", "reason": "asks for slides"}\n```',
    '{"label": "Complaint", "reason": "too fast"}',
]

for number, reply in enumerate(replies, start=1):
    try:
        json.loads(reply)
        print(number, "JSON ok")
    except json.JSONDecodeError:
        print(number, "not JSON")

## Exercise 2: Parse and check the reply

Write `parse_label(reply)` that returns `{"label": ..., "reason": ...}` or `None`:

1. Remove a ```` ```json ```` code fence if the reply starts with ```` ``` ```` (as in the demo).
2. `json.loads` inside `try/except json.JSONDecodeError` → `None` if it fails.
3. The result must be a dict; `label` must be a string that, after `.strip().lower()`, is in `LABELS`;
   `reason` must be a non-empty string. Otherwise → `None`.
4. Return the **cleaned** label (lower case).

Expected output for the six replies:
```
1 {'label': 'praise', 'reason': 'positive about labs'}
2 None
3 None
4 None
5 {'label': 'suggestion', 'reason': 'asks for slides'}
6 {'label': 'complaint', 'reason': 'too fast'}
```

In [ ]:
# your code here


# for number, reply in enumerate(replies, start=1):
#     print(number, parse_label(reply))

## Exercise 3: The one AI function

Write `classify_comment(comment, ask=ask_llm)`:

1. `reply = ask(build_prompt(comment))`
2. If `reply == REFUSAL_MESSAGE` → return `keyword_label(comment)` with `"source": "fallback (declined)"` added.
3. `result = parse_label(reply)`; if `None` → `keyword_label(comment)` with `"source": "fallback (bad reply)"`.
4. Otherwise return `result` with `"source": "ai"`.

Expected output (offline, so the mock answers):
```
{'label': 'suggestion', 'reason': "contains 'please'", 'source': 'ai'}
{'label': 'praise', 'reason': 'no complaint or suggestion words', 'source': 'ai'}
```

In [ ]:
# your code here


# print(classify_comment("Please share the slides earlier."))
# print(classify_comment("Great examples in every lesson."))

## Exercise 4: Test it with fake models

`fake_good` is given. Write three more fakes with the same shape: one parameter, `prompt`, and a string returned:

- `fake_not_json`: returns `"I think this is praise."`
- `fake_refusal`: returns `REFUSAL_MESSAGE`
- `fake_bad_label`: returns valid JSON with the label `"angry"`

Then write four test functions (one per fake) that check the `"source"` of `classify_comment(...)`, call them,
and print `4 tests passed`.

In [ ]:
def fake_good(prompt):
    return '{"label": "complaint", "reason": "room too cold"}'


# your code here

## Exercise 5: A small evaluation

Run `classify_comment` on the labelled comments below. Print every mistake and the accuracy.

Expected output (offline, with the mock):
```
WRONG: 'The room was freezing and the projector kept failing.' expected complaint, got praise
WRONG: 'Not too bad overall, I enjoyed it.' expected praise, got complaint
Accuracy: 6/8
```

Then answer in one sentence each: *What did you just measure offline: the model or something else? How would
you measure the real model?*

In [ ]:
labelled = [
    ("Loved the project weeks, best part of the course!", "praise"),
    ("Please upload the slides before each class.", "suggestion"),
    ("The pace in week 10 was too fast and I got lost.", "complaint"),
    ("It would be great to have more practice exercises.", "suggestion"),
    ("The room was freezing and the projector kept failing.", "complaint"),
    ("Clear explanations, I finally understand functions.", "praise"),
    ("Could we get the homework a day earlier?", "suggestion"),
    ("Not too bad overall, I enjoyed it.", "praise"),
]

# your code here

*Your answers:*

## Exercise 6 (challenge): Option A, a model trained at start-up

1. Write `train_pass_model()` that loads `../../../datasets/students.csv`, trains a `LogisticRegression`
   (`max_iter=1000`) on `study_hours` and `attendance_pct` to predict `passed`, and returns the model.
2. Write `predict_pass(model, study_hours, attendance_pct)`: raise `ValueError` if hours are outside 0–60 or
   attendance outside 0–100; otherwise return `{"passed": True/False, "chance": <probability, 2 decimals>}`.
3. Write tests: bad inputs raise; the chance is between 0 and 1; **more study hours never lower the chance**
   (compare 2 vs 12 hours at 80 % attendance).

Expected output (your numbers may differ slightly in the last decimal):
```
{'passed': False, 'chance': 0.27}
{'passed': True, 'chance': 0.88}
tests passed
```
for `predict_pass(model, 2, 80)` and `predict_pass(model, 12, 80)`.

In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.linear_model import LogisticRegression

# your code here